# Импорт библиотек

In [2]:
import sys
import os

sys.path.append(os.path.abspath('lib'))

from preprocessing_pipeline import create_combined_pipeline
from test_models import run_models_classifications

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score
from scipy.stats import randint, uniform

# Считывание данных

In [4]:
df = pd.read_csv('data/processed.csv')
df.shape

(998, 214)

# Очистка таргета от выбросов

In [6]:
df = df[df['SI'] < 2000]

# Подготовка данных для эксперемента

In [8]:
combined_pipeline = create_combined_pipeline()
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y1 = df['IC50, mM']
y2 = df['CC50, mM']
y3 = df['SI']
X_transformed = combined_pipeline.fit_transform(X)
df = pd.concat([X_transformed, y1, y2, y3], axis=1)

In [9]:
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y = df['SI'].apply(lambda v: 1 if v >= 8 else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train dataset size: {X_train.shape}, {y_train.shape}')
print(f'Test dataset size: {X_test.shape}, {y_test.shape}')

Train dataset size: (794, 107), (794,)
Test dataset size: (199, 107), (199,)


# Эксперемент с моделями

In [11]:
run_models_classifications(X_train, X_test, y_train, y_test)

,Model,WA f1,accuracy,WA precision,WA recall,WA support,roc_auc
6,LightGBM,0.75,0.76,0.75,0.76,199.0,0.74
3,Random Forest,0.73,0.74,0.73,0.74,199.0,0.76
4,XGBoost,0.73,0.74,0.73,0.74,199.0,0.73
9,AdaBoost,0.73,0.74,0.74,0.74,199.0,0.74
5,Gradient Boosting,0.72,0.73,0.73,0.73,199.0,0.73
7,CatBoost,0.72,0.73,0.73,0.73,199.0,0.75
0,Logistic Regression,0.71,0.72,0.71,0.72,199.0,0.74
8,HistGradientBoosting,0.71,0.73,0.72,0.73,199.0,0.74
1,Decision Tree,0.69,0.69,0.68,0.69,199.0,0.65
2,KNeighbors,0.67,0.69,0.68,0.69,199.0,0.70


# Подбор гиперпараметров

In [57]:
param_dist = {
    'n_estimators': randint(50, 300),
    'num_leaves': randint(20, 50),
    'max_depth': randint(3, 15),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0, 1),
    'min_child_samples': randint(5, 20),
    'min_split_gain': uniform(0, 0.1),
}
random_search = RandomizedSearchCV(
    estimator=LGBMClassifier(verbose=-1, random_state=42),
    param_distributions=param_dist,
    n_iter=200,
    scoring='roc_auc',
    cv=5,
    verbose=0,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

best_params = random_search.best_params_
print("Лучшие параметры:", best_params)

Лучшие параметры: {'colsample_bytree': 0.76821431146603, 'learning_rate': 0.24540047918329286, 'max_depth': 4, 'min_child_samples': 15, 'min_split_gain': 0.021296416150891076, 'n_estimators': 215, 'num_leaves': 47, 'reg_alpha': 0.9804627250923779, 'reg_lambda': 0.6080878495418852, 'subsample': 0.854657728648913}


In [62]:
model = LGBMClassifier(
    verbose=-1,
    colsample_bytree=0.77,
    learning_rate=0.245,
    max_depth=4,
    min_child_samples=15,
    min_split_gain=0.02,
    n_estimators=215,
    num_leaves=47,
    reg_alpha=0.98,
    reg_lambda=0.6,
    subsample=0.85,
    random_state=42,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()
roc_auc = roc_auc_score(y_test, y_score)

print("Accuracy:", round(report['accuracy'], 2))
print("ROC AUC Score:", round(roc_auc, 2))
report_df

Accuracy: 0.77
ROC AUC Score: 0.75


,precision,recall,f1-score,support
0,0.785235,0.900000,0.838710,130.000000
1,0.740000,0.536232,0.621849,69.000000
accuracy,0.773869,0.773869,0.773869,0.773869
macro avg,0.762617,0.718116,0.730279,199.000000
weighted avg,0.769550,0.773869,0.763517,199.000000
